# Learnings 02: Complete Project Workflow

This notebook explains the full project workflow in simple language, from raw arXiv papers to hybrid search, RAG answers, agentic answers, tracing, and Telegram usage.

This notebook is about understanding the pipeline as one connected system.

## 1. The Main Goal Of The Project

The project builds a research-paper assistant around arXiv papers, especially from the `cs.AI` category.

The high-level goal is:
- fetch recent papers
- parse and store them
- index them for search
- answer questions over them using RAG
- add an agentic workflow for more intelligent retrieval behavior
- expose the system via API and Telegram
- monitor the system with Langfuse

## 2. Infrastructure And Startup Flow

At startup, Docker Compose brings up the main services:
- Postgres
- Redis
- OpenSearch
- Ollama
- Airflow
- API
- Telegram bot
- Langfuse stack

The API startup sequence is roughly:
1. load settings from `.env`
2. initialize database connection
3. initialize OpenSearch and verify the index
4. initialize arXiv client, parser, embeddings, Ollama, cache, and tracing
5. start serving requests

The Telegram bot now runs as a dedicated service instead of being started inside every API worker. This was an important deployment correction.

## 3. Data Ingestion Workflow

The ingestion pipeline begins with arXiv metadata retrieval.

### Step 1: Fetch metadata
The arXiv service queries the arXiv API and gets recent papers from the configured category.

Important fields include:
- arXiv ID
- title
- authors
- abstract
- publish date
- PDF URL

### Step 2: Download PDFs
Each paper's PDF is downloaded and cached locally. We improved this step so the downloader:
- follows redirects
- retries on failure
- writes through a temporary file first
- reduces the chance of corrupted cached files

### Step 3: Parse PDFs
The parser stack reads the cached PDF and extracts text.

Final behavior after debugging:
- normal path uses Docling
- long PDFs are truncated to the first configured pages instead of skipped entirely
- if Docling fails on a problematic PDF, a fallback parser can rescue the text
- invalid cached files can trigger a fresh redownload

### Step 4: Store into PostgreSQL
After parsing, the project stores the paper record plus extracted text in Postgres.

This gives us a persistent source-of-truth database for paper metadata and parsed content.

## 4. Indexing Workflow

Once parsed text exists, the indexing pipeline prepares it for search.

### Step 1: Chunk the text
Each paper's text is split into chunks so that search and embeddings work on manageable pieces.

Why chunking matters:
- more precise retrieval
- better semantic matching
- less noise than searching one giant document

### Step 2: Generate embeddings with Jina
Each chunk is sent to Jina to produce a vector representation.

This allows semantic retrieval later.

### Step 3: Store chunks in OpenSearch
Each chunk, its metadata, and its embedding are indexed into OpenSearch.

OpenSearch becomes the retrieval engine that powers:
- keyword search
- BM25 ranking
- vector search
- hybrid search

After debugging and retries, this indexing stage now works end-to-end.

## 5. Search And Normal RAG Workflow

### Hybrid Search Endpoint
The `/api/v1/hybrid-search` endpoint takes a query and returns the most relevant indexed chunks.

Search flow:
1. embed the query with Jina
2. send keyword + vector search to OpenSearch
3. combine results through hybrid ranking
4. return top matching chunks

### Normal `/ask` RAG Endpoint
The normal RAG flow is:
1. receive the user question
2. check exact-match cache in Redis
3. retrieve relevant chunks from OpenSearch
4. build a RAG prompt with those chunks
5. ask Ollama to generate an answer
6. return answer + sources + chunk count

This path turned out to be concise and strong after the debugging phase.

## 6. Streaming RAG Workflow

The `/api/v1/stream` endpoint follows the same retrieval logic as normal RAG, but instead of returning the answer all at once, it streams partial output.

Why streaming matters:
- the user sees progress earlier
- long answers feel more interactive
- it improves perceived responsiveness

## 7. Agentic RAG Workflow

The agentic system adds decision-making on top of retrieval.

Current agentic workflow:
1. validate whether the query is in scope
2. retrieve candidate documents
3. grade document relevance
4. optionally rewrite the query if needed
5. generate the final answer from context
6. return sources, reasoning steps, retrieval attempt count, and trace id

This workflow is built with LangGraph-style nodes and state passing.

The final agentic response now includes:
- grounded answer
- normalized source URLs
- actual retrieved chunk count
- reasoning steps
- Langfuse trace id

## 8. Telegram Bot Workflow

The Telegram bot provides a chat-based interface to the same knowledge system.

Typical bot flow:
1. user sends a message in Telegram
2. bot converts it into a request-like structure
3. bot runs search + RAG logic
4. bot replies with answer and sources

Important deployment lesson:
- the bot should run as a dedicated single service
- it should not be started once per API worker

That correction was necessary because Telegram polling allows only one active consumer for a bot token.

## 9. Monitoring And Tracing Workflow

Langfuse is now integrated into the project.

Current usage:
- traces for agentic requests
- request-level metadata
- LLM callback traces
- trace ids returned to the client

Why this matters:
- helps inspect why answers are good or bad
- helps compare normal and agentic RAG
- helps debug retrieval and prompting

## 10. Final Big Picture

The complete working pipeline can be summarized as:

arXiv API -> PDF download -> PDF parsing -> PostgreSQL storage -> chunking -> Jina embeddings -> OpenSearch indexing -> hybrid retrieval -> Ollama answer generation -> API / Telegram response -> Langfuse trace

This is the full operational system we built and stabilized during the project.